# Phase 2 (IHDS-II) — Feature Engineering

[`walkthrough/ihds_phase2.md`](../walkthrough/ihds_phase2.md)

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../dataset/ihds2_households.csv")

EXPENSE_CATEGORIES = [
    "Groceries", "Eating_Out", "Utilities", "Rent", "Transport", "Healthcare",
    "Education", "Entertainment", "Insurance", "Clothing_Footwear", "Miscellaneous",
]
SHARE_COLS = [f"{c}_Share" for c in EXPENSE_CATEGORIES]
CV = StratifiedKFold(5, shuffle=True, random_state=42)

zero_rate = (df[SHARE_COLS] == 0).mean().sort_values()
print("zero-rate by share (Phase 1 finding):")
print(zero_rate.round(4).to_string())
print(f"\nShape: {df.shape}")

zero-rate by share (Phase 1 finding):
Groceries_Share            0.0006
Miscellaneous_Share        0.0007
Utilities_Share            0.0016
Clothing_Footwear_Share    0.0139
Transport_Share            0.1141
Healthcare_Share           0.1915
Education_Share            0.3590
Entertainment_Share        0.6912
Eating_Out_Share           0.7196
Insurance_Share            0.7367
Rent_Share                 0.9045

Shape: (41518, 50)


## Q1 — Do expense-to-income ratios generalise better across income levels than raw expense values?

The original Phase 2 tested this against `Goal_Met`. That is not possible here: Phase 1 showed
raw rupee categories reconstruct `Goal_Met` with 99.75% agreement, so any comparison against it
would rank the leaky representation first by construction. The comparison is run against
`Has_Bank_Savings` instead — a survey-reported behaviour that is not an accounting function of
the expense columns, so both representations face it on equal terms.

In [2]:
neutral = df[df["Has_Bank_Savings"].notna()].copy()
y_neutral = neutral["Has_Bank_Savings"].astype(int)
print(f"neutral target: Has_Bank_Savings, n={len(neutral):,}, positive rate={y_neutral.mean():.4f}")

REPRESENTATIONS = {
    "raw rupees": EXPENSE_CATEGORIES,
    "expense / income": None,      # built below
    "share of expenditure": SHARE_COLS,
}
ratios = neutral[EXPENSE_CATEGORIES].div(neutral["INCOME"], axis=0)
ratios.columns = [f"{c}_Ratio" for c in EXPENSE_CATEGORIES]
neutral = pd.concat([neutral, ratios], axis=1)
REPRESENTATIONS["expense / income"] = list(ratios.columns)

rows = []
for name, cols in REPRESENTATIONS.items():
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(max_iter=2000))
    res = cross_validate(model, neutral[cols], y_neutral, cv=CV, scoring=["roc_auc", "f1_macro"])
    rows.append({"representation": name,
                 "ROC-AUC": res["test_roc_auc"].mean(),
                 "F1 (macro)": res["test_f1_macro"].mean()})
print("\nSame-distribution CV (all income levels mixed):")
print(pd.DataFrame(rows).set_index("representation").round(4).to_string())

neutral target: Has_Bank_Savings, n=41,385, positive rate=0.5759



Same-distribution CV (all income levels mixed):
                      ROC-AUC  F1 (macro)
representation                           
raw rupees             0.6618      0.5988
expense / income       0.6303      0.3939
share of expenditure   0.6491      0.5878


In [3]:
# The actual question is generalisation ACROSS income levels: fit on one half of the
# income distribution, score on the other. A representation that merely re-encodes income
# will transfer badly; one that captures behaviour should transfer.
median_income = neutral["INCOME"].median()
low = neutral["INCOME"] <= median_income
rows = []
for name, cols in REPRESENTATIONS.items():
    for train_label, train_mask in [("low -> high", low), ("high -> low", ~low)]:
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(max_iter=2000))
        model.fit(neutral.loc[train_mask, cols], y_neutral[train_mask])
        from sklearn.metrics import roc_auc_score
        score = roc_auc_score(y_neutral[~train_mask],
                              model.predict_proba(neutral.loc[~train_mask, cols])[:, 1])
        rows.append({"representation": name, "transfer": train_label, "ROC-AUC": score})

transfer = pd.DataFrame(rows).pivot(index="representation", columns="transfer", values="ROC-AUC")
transfer["mean"] = transfer.mean(axis=1)
print("Cross-income-level transfer (trained on one half, scored on the other):")
print(transfer.round(4).to_string())

Cross-income-level transfer (trained on one half, scored on the other):
transfer              high -> low  low -> high    mean
representation                                        
expense / income           0.5828       0.5957  0.5893
raw rupees                 0.6063       0.6428  0.6245
share of expenditure       0.6075       0.6275  0.6175


## Q2 / Decision 1 — How should categorical features be encoded, and what about missing categories?

In [4]:
CATEGORICALS = ["Occupation", "Area_Type", "Caste_Group", "Religion"]
print("cardinality and missingness:")
for c in CATEGORICALS:
    print(f"  {c:14s} levels={df[c].nunique():2d}  missing={df[c].isna().sum():4d} ({df[c].isna().mean():.3%})")

# Is missingness informative? If households refusing to state caste differ in outcome from
# the population, an explicit Unknown level preserves that; imputation erases it.
print("\nGoal_Met rate by whether the category is missing:")
for c in ["Caste_Group", "Religion"]:
    miss = df[c].isna()
    if miss.sum():
        print(f"  {c:14s} missing: {df.loc[miss, 'Goal_Met'].mean():.4f} (n={miss.sum():3d})   "
              f"present: {df.loc[~miss, 'Goal_Met'].mean():.4f}")
print(f"\noverall Goal_Met rate: {df['Goal_Met'].mean():.4f}")

for c in CATEGORICALS:
    df[c] = df[c].fillna("Unknown")
print("\nafter fillna('Unknown'):", {c: int(df[c].isna().sum()) for c in CATEGORICALS})

cardinality and missingness:
  Occupation     levels= 6  missing=   0 (0.000%)
  Area_Type      levels= 4  missing=   0 (0.000%)
  Caste_Group    levels= 6  missing=  85 (0.205%)
  Religion       levels= 8  missing=  12 (0.029%)

Goal_Met rate by whether the category is missing:
  Caste_Group    missing: 0.3176 (n= 85)   present: 0.3193
  Religion       missing: 0.3333 (n= 12)   present: 0.3193

overall Goal_Met rate: 0.3193

after fillna('Unknown'): {'Occupation': 0, 'Area_Type': 0, 'Caste_Group': 0, 'Religion': 0}


## Decision 2 — Zero-inflation: should the majority-zero categories get explicit participation indicators?

In [5]:
# Phase 1: four shares are zero for the majority of households. A single share column
# conflates "spends nothing on this" with "spends a little", which are different states.
ZERO_INFLATED = [c for c in SHARE_COLS if (df[c] == 0).mean() > 0.30]
print("categories treated as semi-continuous (>30% zero):")
for c in ZERO_INFLATED:
    print(f"  {c:26s} zero in {(df[c] == 0).mean():.1%} of households")

INDICATORS = []
for c in ZERO_INFLATED:
    name = f"Spends_On_{c.replace('_Share', '')}"
    df[name] = (df[c] > 0).astype(int)
    INDICATORS.append(name)

print("\nGoal_Met rate by participation:")
for ind in INDICATORS:
    g = df.groupby(ind)["Goal_Met"].agg(["mean", "count"])
    lift = g.loc[1, "mean"] - g.loc[0, "mean"]
    print(f"  {ind:28s} no={g.loc[0, 'mean']:.4f}  yes={g.loc[1, 'mean']:.4f}  lift={lift:+.4f}")

categories treated as semi-continuous (>30% zero):
  Eating_Out_Share           zero in 72.0% of households
  Rent_Share                 zero in 90.4% of households
  Education_Share            zero in 35.9% of households
  Entertainment_Share        zero in 69.1% of households
  Insurance_Share            zero in 73.7% of households

Goal_Met rate by participation:
  Spends_On_Eating_Out         no=0.3067  yes=0.3515  lift=+0.0448
  Spends_On_Rent               no=0.3198  yes=0.3148  lift=-0.0050
  Spends_On_Education          no=0.3721  yes=0.2897  lift=-0.0825
  Spends_On_Entertainment      no=0.3125  yes=0.3346  lift=+0.0221
  Spends_On_Insurance          no=0.2938  yes=0.3906  lift=+0.0968


In [6]:
# Do the indicators add anything the shares do not already carry?
base_num = SHARE_COLS + ["Log_Income", "Household_Size", "Dependents", "Head_Age", "Max_Adult_Education"]
y = df["Goal_Met"]


def score(num_cols, cat_cols=CATEGORICALS, label="", model="logreg"):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    est = (LogisticRegression(max_iter=2000, class_weight="balanced") if model == "logreg"
           else RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight="balanced"))
    res = cross_validate(make_pipeline(pre, est), df[num_cols + cat_cols], y, cv=CV,
                         scoring=["roc_auc", "f1_macro"])
    return {"feature set": label, "model": model,
            "ROC-AUC": res["test_roc_auc"].mean(), "F1 (macro)": res["test_f1_macro"].mean()}


rows = [score(base_num, label="shares only"),
        score(base_num + INDICATORS, label="shares + participation indicators")]
print(pd.DataFrame(rows).set_index("feature set").round(4).to_string())

                                    model  ROC-AUC  F1 (macro)
feature set                                                   
shares only                        logreg   0.9183      0.8160
shares + participation indicators  logreg   0.9204      0.8195


## Decision 3 — `Debt_To_Income`: winsorise, log, or leave raw?

In [7]:
print("raw Debt_To_Income tail:")
print(df["Debt_To_Income"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(3).to_string())

p99 = df["Debt_To_Income"].quantile(0.99)
df["Debt_To_Income_W"] = df["Debt_To_Income"].clip(upper=p99)
df["Log_Debt_To_Income"] = np.log1p(df["Debt_To_Income"])
df["Has_Debt"] = (df["Debt_To_Income"] > 0).astype(int)
print(f"\nwinsorised at 99th percentile = {p99:.3f}; affects {(df['Debt_To_Income'] > p99).sum():,} households")

rows = []
for variant in ["Debt_To_Income", "Debt_To_Income_W", "Log_Debt_To_Income"]:
    for m in ["logreg", "rf"]:
        r = score(base_num + INDICATORS + [variant, "Has_Debt"], label=variant, model=m)
        rows.append(r)
print("\n" + pd.DataFrame(rows).pivot(index="feature set", columns="model",
                                       values=["ROC-AUC", "F1 (macro)"]).round(4).to_string())

raw Debt_To_Income tail:
count    41518.000
mean         0.883
std         11.750
min          0.000
50%          0.000
90%          1.429
99%         10.234
99.9%       66.667
max       1300.000

winsorised at 99th percentile = 10.234; affects 416 households



                   ROC-AUC         F1 (macro)        
model               logreg      rf     logreg      rf
feature set                                          
Debt_To_Income      0.9210  0.9172     0.8201  0.8176
Debt_To_Income_W    0.9212  0.9168     0.8202  0.8179
Log_Debt_To_Income  0.9215  0.9172     0.8203  0.8176


## Decision 4 — Compositional transform: making the shares usable by distance-based methods

Phase 1 established that the 11 shares live on a simplex (they sum to 1), so Euclidean distance
between them is not well defined — which Phase 6's clustering depends on. The standard fix is a
centred log-ratio (CLR) transform, but `log(0)` is undefined and four categories are majority-zero.

In [8]:
# Restrict the log-ratio transform to a CORE SUB-COMPOSITION: the parts that are
# present for nearly every household. Zero-inflated categories stay as share +
# participation indicator instead of being forced through a log.
CORE = [c for c in SHARE_COLS if (df[c] == 0).mean() < 0.20]
print("core sub-composition (<20% zero):")
for c in CORE:
    print(f"  {c:26s} zero in {(df[c] == 0).mean():.2%}")
print("\nexcluded from the log-ratio transform:", [c.replace("_Share", "") for c in SHARE_COLS if c not in CORE])


def multiplicative_replacement(X):
    """Replace zeros per Martin-Fernandez et al.: delta = 0.65x the smallest observed
    positive value in that part, then rescale the non-zeros so rows still sum to 1."""
    A = X.to_numpy(dtype=float).copy()
    deltas = np.array([X[c][X[c] > 0].min() * 0.65 for c in X.columns])
    print("per-part zero-replacement deltas:")
    for c, d in zip(X.columns, deltas):
        print(f"  {c:26s} {d:.3e}")
    row_sum = A.sum(axis=1, keepdims=True)
    A = np.divide(A, row_sum, out=np.full_like(A, np.nan), where=row_sum > 0)
    zeros = A == 0
    wide = np.broadcast_to(deltas, A.shape)
    lost = (zeros * wide).sum(axis=1, keepdims=True)
    return np.where(zeros, wide, A * (1 - lost))


all_zero = df[CORE].sum(axis=1) == 0
print(f"households spending nothing on ANY core category: {all_zero.sum()}"
      "  -> CLR undefined for these, left as NaN")

core_closed = multiplicative_replacement(df[CORE])
log_core = np.log(core_closed)
clr = log_core - log_core.mean(axis=1, keepdims=True)
CLR_COLS = [c.replace("_Share", "_CLR") for c in CORE]
df[CLR_COLS] = clr

valid = ~np.isnan(clr).any(axis=1)
print(f"\nCLR rows sum to zero (defining property): "
      f"max |row sum| = {np.abs(clr[valid].sum(axis=1)).max():.2e}")
print("\nCLR feature summary:")
print(pd.DataFrame(clr, columns=CLR_COLS).describe().T[["mean", "std", "min", "max"]].round(3).to_string())

core sub-composition (<20% zero):
  Groceries_Share            zero in 0.06%
  Utilities_Share            zero in 0.16%
  Transport_Share            zero in 11.41%
  Healthcare_Share           zero in 19.15%
  Clothing_Footwear_Share    zero in 1.39%
  Miscellaneous_Share        zero in 0.07%

excluded from the log-ratio transform: ['Eating_Out', 'Rent', 'Education', 'Entertainment', 'Insurance']
households spending nothing on ANY core category: 2  -> CLR undefined for these, left as NaN
per-part zero-replacement deltas:
  Groceries_Share            2.230e-03
  Utilities_Share            2.505e-04
  Transport_Share            6.738e-05
  Healthcare_Share           8.302e-05
  Clothing_Footwear_Share    1.129e-04
  Miscellaneous_Share        6.776e-04

CLR rows sum to zero (defining property): max |row sum| = 7.99e-15

CLR feature summary:
                        mean    std    min    max
Groceries_CLR          1.954  0.776 -2.679  7.279
Utilities_CLR          0.313  0.878 -5.381  6.913

In [9]:
# Does the CLR representation actually help? Compare on a distance-sensitive model
# (logistic regression is linear in the features; the CLR changes what "linear" means).
rows = [
    score(base_num + INDICATORS + ["Debt_To_Income_W", "Has_Debt"], label="raw shares", model="logreg"),
    score([c for c in base_num if c not in CORE] + CLR_COLS + INDICATORS
          + ["Debt_To_Income_W", "Has_Debt"], label="CLR core + zero-inflated shares", model="logreg"),
    score(base_num + INDICATORS + ["Debt_To_Income_W", "Has_Debt"], label="raw shares", model="rf"),
    score([c for c in base_num if c not in CORE] + CLR_COLS + INDICATORS
          + ["Debt_To_Income_W", "Has_Debt"], label="CLR core + zero-inflated shares", model="rf"),
]
print(pd.DataFrame(rows).pivot(index="feature set", columns="model",
                               values=["ROC-AUC", "F1 (macro)"]).round(4).to_string())

                                ROC-AUC         F1 (macro)        
model                            logreg      rf     logreg      rf
feature set                                                       
CLR core + zero-inflated shares  0.9117  0.9095     0.8097  0.8114
raw shares                       0.9212  0.9168     0.8202  0.8179


### Verdict on the compositional transform

The CLR representation is **rejected for the classification pipeline**. It scores worse than the
raw shares on both model families, and the VIF table below shows why it cannot go into a linear
model at all: CLR components sum to zero by construction, so their covariance matrix is singular.
It is still exported for Phase 6 to evaluate -- clustering is the step that genuinely needs a
metric on the simplex -- but an isometric log-ratio (ILR) basis, which drops one dimension and is
therefore non-singular, is the better candidate there.

## Q3 — Which features require scaling, and does that depend on the downstream model?

In [10]:
df["Dependency_Ratio"] = df["Dependents"] / df["Household_Size"]
# Raw shares, not CLR: the comparison above rejected the log-ratio transform.
FINAL_NUM = base_num + INDICATORS + ["Debt_To_Income_W", "Has_Debt", "Dependency_Ratio"]

spread = df[FINAL_NUM].agg(["min", "max", "std", "skew"]).T
spread["range"] = spread["max"] - spread["min"]
print("Scale and skew across the final numeric features:")
print(spread.sort_values("range", ascending=False).round(3).to_string())
print(f"\nratio of largest to smallest standard deviation: {spread['std'].max() / spread['std'].min():,.0f}x")

Scale and skew across the final numeric features:
                            min     max     std   skew   range
Head_Age                 11.000  99.000  13.566  0.285  88.000
Household_Size            1.000  33.000   2.316  1.431  32.000
Dependents                0.000  19.000   1.566  1.113  19.000
Max_Adult_Education       0.000  16.000   5.093 -0.271  16.000
Log_Income                4.605  16.246   1.061 -0.316  11.640
Debt_To_Income_W          0.000  10.234   1.468  4.593  10.234
Groceries_Share           0.000   1.000   0.150 -0.204   1.000
Utilities_Share           0.000   1.000   0.057  1.645   1.000
Spends_On_Entertainment   0.000   1.000   0.462  0.828   1.000
Spends_On_Insurance       0.000   1.000   0.440  1.075   1.000
Has_Debt                  0.000   1.000   0.497  0.213   1.000
Spends_On_Rent            0.000   1.000   0.294  2.753   1.000
Dependency_Ratio          0.000   1.000   0.262  0.246   1.000
Rent_Share                0.000   1.000   0.045  6.787   1.000
Spend

In [11]:
# Scaling matters for the linear model and not for the tree; and given the skew,
# RobustScaler is the safer default over StandardScaler.
rows = []
for scaler_name, scaler in [("none", "passthrough"), ("StandardScaler", StandardScaler()),
                            ("RobustScaler", RobustScaler())]:
    for m in ["logreg", "rf"]:
        pre = ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", scaler)]), FINAL_NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICALS),
        ])
        est = (LogisticRegression(max_iter=2000, class_weight="balanced") if m == "logreg"
               else RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight="balanced"))
        res = cross_validate(make_pipeline(pre, est), df[FINAL_NUM + CATEGORICALS], y,
                             cv=CV, scoring=["roc_auc"])
        rows.append({"scaler": scaler_name, "model": m, "ROC-AUC": res["test_roc_auc"].mean()})
print(pd.DataFrame(rows).pivot(index="scaler", columns="model", values="ROC-AUC").round(4).to_string())

model           logreg      rf
scaler                        
RobustScaler    0.9212  0.9162
StandardScaler  0.9212  0.9160
none            0.9211  0.9160


## Q4 — Are any features redundant or highly collinear?

In [12]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_num = df[FINAL_NUM].fillna(df[FINAL_NUM].median())
X_std = (X_num - X_num.mean()) / X_num.std()
X_std = X_std.loc[:, X_std.std() > 0]
vif = pd.Series(
    [variance_inflation_factor(X_std.to_numpy(), i) for i in range(X_std.shape[1])],
    index=X_std.columns,
).sort_values(ascending=False)
print("Variance inflation factors (>10 conventionally indicates a problem):")
print(vif.round(2).to_string())

cm = X_num.corr().abs()
pairs = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("\nMost correlated feature pairs:")
print(pairs.head(8).round(4).to_string())

Variance inflation factors (>10 conventionally indicates a problem):
Groceries_Share             inf
Eating_Out_Share            inf
Utilities_Share             inf
Rent_Share                  inf
Transport_Share             inf
Healthcare_Share            inf
Education_Share             inf
Entertainment_Share         inf
Insurance_Share             inf
Clothing_Footwear_Share     inf
Miscellaneous_Share         inf
Dependents                 8.98
Household_Size             5.23
Dependency_Ratio           4.61
Spends_On_Insurance        1.89
Spends_On_Rent             1.86
Spends_On_Eating_Out       1.79
Log_Income                 1.68
Spends_On_Education        1.55
Max_Adult_Education        1.52
Spends_On_Entertainment    1.46
Debt_To_Income_W           1.34
Has_Debt                   1.27
Head_Age                   1.17

Most correlated feature pairs:
Household_Size       Dependents                 0.7142
Dependents           Dependency_Ratio           0.6776
Rent_Share           

## Final feature set

In [13]:
FEATURE_COLS = FINAL_NUM + CATEGORICALS
# CLR columns are exported but excluded from FEATURE_COLS - reserved for Phase 6.
out = df[["IDHH", "WT"] + FEATURE_COLS + CLR_COLS + ["Goal_Met", "Savings_Rate"]].copy()
out.to_csv("../dataset/ihds2_features.csv", index=False)

print(f"engineered feature matrix: {len(FEATURE_COLS)} features, {len(out):,} households")
print(f"  numeric     ({len(FINAL_NUM):2d}): {FINAL_NUM}")
print(f"  categorical ({len(CATEGORICALS):2d}): {CATEGORICALS}")
print("\nfinal cross-validated performance:")
print(pd.DataFrame([score(FINAL_NUM, label="final", model="logreg"),
                    score(FINAL_NUM, label="final", model="rf")]).set_index("model").round(4).to_string())
print("\nwrote ../dataset/ihds2_features.csv")

engineered feature matrix: 28 features, 41,518 households
  numeric     (24): ['Groceries_Share', 'Eating_Out_Share', 'Utilities_Share', 'Rent_Share', 'Transport_Share', 'Healthcare_Share', 'Education_Share', 'Entertainment_Share', 'Insurance_Share', 'Clothing_Footwear_Share', 'Miscellaneous_Share', 'Log_Income', 'Household_Size', 'Dependents', 'Head_Age', 'Max_Adult_Education', 'Spends_On_Eating_Out', 'Spends_On_Rent', 'Spends_On_Education', 'Spends_On_Entertainment', 'Spends_On_Insurance', 'Debt_To_Income_W', 'Has_Debt', 'Dependency_Ratio']
  categorical ( 4): ['Occupation', 'Area_Type', 'Caste_Group', 'Religion']

final cross-validated performance:


       feature set  ROC-AUC  F1 (macro)
model                                  
logreg       final   0.9212      0.8203
rf           final   0.9160      0.8162

wrote ../dataset/ihds2_features.csv
